# Campus Chatbot - Model Training
Run this notebook once. Models are saved to disk and never need retraining unless you change your data.

In [ ]:
import sys
sys.path.insert(0, '..')
import os, json, random, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
warnings.filterwarnings('ignore')
BASE_DIR = Path('..')
print('Setup complete')

In [ ]:
# ── CELL 2: Load Knowledge Base ──────────────────────────────────────
with open(BASE_DIR / 'data/kb/knowledge_base.json', 'r') as f:
    kb = json.load(f)
print(f'KB loaded: {len(kb)} records')
places = [loc['name'] for loc in kb]
departments = [loc['name'].replace(' Department','') for loc in kb if loc.get('category') == 'department']
print(f'Places: {len(places)}, Departments: {len(departments)}')

In [ ]:
# ── CELL 3: Build Intent Dataset ─────────────────────────────────────
INTENTS = [
    'find_location', 'ask_hours', 'ask_event', 'menu_query',
    'faculty_query', 'service_query', 'recommend_place',
    'lost_found', 'emergency', 'facility_info',
    'ask_contact', 'ask_department', 'fallback'
]

templates = {
    'find_location': [
        'where is {place}', 'how do i get to {place}', 'take me to {place}',
        'where is the cafeteria', 'where is the library', 'where is the canteen',
        'show me the way to {place}', 'which floor is {place}', 'location of {place}',
        'where can i find {place}', 'directions to {place}', 'how to reach {place}',
        'from civil to mechanical', 'from library to cafeteria', 'from cse to gym',
        'how do i go from {place} to the library', 'navigate to {place}'
    ],
    'ask_hours': [
        'when is {place} open', 'what time does {place} close',
        'is {place} open now', 'is the library open now',
        'opening hours of {place}', 'what are the timings of {place}',
        'is cafeteria open on sunday', 'what time does the gym open',
        'can i visit {place} now', 'is {place} open today'
    ],
    'ask_event': [
        'any events today', 'what workshops are happening',
        'any seminar this week', 'show me upcoming events',
        'anything happening on campus', 'any guest lecture today',
        'events at student union', 'what is on today'
    ],
    'menu_query': [
        'what is today menu', 'price of tea', 'how much is coffee',
        'where can i get lunch', 'what food is available now',
        'what is the price of tea', 'tea price', 'coffee price',
        'what snacks are available', 'price of samosa',
        'what is on the menu today', 'can i pay with student card',
        'how much does lunch cost', 'menu for today', 'friday special menu'
    ],
    'faculty_query': [
        'where is {department} staff room', 'where is {department} faculty room',
        'faculty room for {department}', 'staff room for {department}',
        'where can i meet my professor', 'where do lecturers sit',
        'where is the hod office of {department}', 'where can i meet my project supervisor',
        'where do teachers sit in {department}'
    ],
    'service_query': [
        'where can i charge my laptop', 'where can i print assignment',
        'where can i issue books', 'where can i return books',
        'where can i pay fees', 'my wifi is not working',
        'where can i scan documents', 'where can i get bonafide certificate',
        'where is the it helpdesk', 'how to get student id reissued'
    ],
    'recommend_place': [
        'where can i study peacefully', 'where can i study quietly',
        'suggest a quiet study place', 'where can i relax after class',
        'where can i sit with laptop', 'where can i prepare for exams',
        'i need a peaceful place to work', 'best place to study on campus'
    ],
    'lost_found': [
        'lost my watch', 'lost my wallet', 'lost my laptop',
        'i lost my student id card', 'where is lost and found',
        'where should i report lost item', 'i found someone phone',
        'i lost my bag', 'lost property office'
    ],
    'emergency': [
        'i need first aid', 'where is first aid', 'where is medical room',
        'i feel sick', 'someone is injured', 'i need emergency help',
        'i cut my finger', 'i hurt my hand', 'i am bleeding',
        'i need bandage', 'where can i get medical help',
        'i feel dizzy', 'i have fever', 'i have stomachache',
        'i injured myself', 'i need a doctor'
    ],
    'facility_info': [
        'tell me about {place}', 'what facilities are in {place}',
        'what services does {place} provide', 'what is {place} used for',
        'give me details about {place}', 'what does {place} offer'
    ],
    'ask_contact': [
        'who should i contact for {place}', 'where can visitors ask for help',
        'who handles hostel issues', 'where can i ask general questions',
        'who manages document queries', 'who is the contact for {place}'
    ],
    'ask_department': [
        'tell me about {department}', 'where is {department} department',
        'what labs are in {department}', 'which floor is {department} department',
        'what does {department} department offer'
    ],
    'fallback': [
        'what is the weather', 'tell me a joke', 'who won the match',
        'how do i cook pasta', 'what is bitcoin price', 'book me a flight',
        'what movie should i watch', 'who is the president',
        'play some music', 'what is the news today'
    ]
}

rows = []
SAMPLES = 300
for intent, tmpl_list in templates.items():
    for _ in range(SAMPLES):
        t = random.choice(tmpl_list)
        q = t.format(
            place=random.choice(places),
            department=random.choice(departments) if departments else 'Computer Science'
        )
        rows.append({'query': q, 'intent': intent})

df = pd.DataFrame(rows)
df.to_csv(BASE_DIR / 'data/text/intent_dataset.csv', index=False)
print(f'Dataset: {len(df)} rows, {df.intent.nunique()} intents')
print(df.intent.value_counts())

In [ ]:
# ── CELL 4: Visualise Dataset ─────────────────────────────────────────
df['query_len'] = df['query'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, y='intent', order=df['intent'].value_counts().index, ax=axes[0])
axes[0].set_title('Intent Distribution')
axes[0].set_xlabel('Count')

sns.histplot(df['query_len'], bins=20, kde=True, ax=axes[1])
axes[1].set_title('Query Length Distribution')
axes[1].set_xlabel('Words')

plt.tight_layout()
plt.savefig(BASE_DIR / 'outputs/plots/dataset_overview.png', dpi=150)
plt.show()
print('Plot saved')

In [ ]:
# ── CELL 5: Encode Labels and Split ──────────────────────────────────
le = LabelEncoder()
df['label'] = le.fit_transform(df['intent'])

id2label = {i: l for i, l in enumerate(le.classes_)}
label2id = {l: i for i, l in id2label.items()}

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_df.to_csv(BASE_DIR / 'data/text/train.csv', index=False)
test_df.to_csv(BASE_DIR  / 'data/text/test.csv',  index=False)
with open(BASE_DIR / 'models/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
with open(BASE_DIR / 'models/id2label.json', 'w') as f:
    json.dump(id2label, f)

print(f'Train: {len(train_df)}, Test: {len(test_df)}')
print('Label mapping:', id2label)

In [ ]:
# ── CELL 6: Tokenise for DistilBERT ──────────────────────────────────
import datasets.config
datasets.config.TORCHVISION_AVAILABLE = False

from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_df[['query','label']])
test_ds  = Dataset.from_pandas(test_df[['query','label']])

def tokenize(batch):
    return tokenizer(batch['query'], truncation=True, padding='max_length', max_length=64)

train_ds = train_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize,  batched=True)

train_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])
test_ds.set_format( type='torch', columns=['input_ids','attention_mask','label'])
print('Tokenisation complete')

In [ ]:
# ── CELL 7: Train DistilBERT Intent Classifier ───────────────────────
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(le.classes_), id2label=id2label, label2id=label2id
)

def compute_metrics(ep):
    preds = np.argmax(ep.predictions, axis=1)
    p, r, f, _ = precision_recall_fscore_support(ep.label_ids, preds, average='weighted', zero_division=0)
    return {'accuracy': accuracy_score(ep.label_ids, preds), 'precision': p, 'recall': r, 'f1': f}

args = TrainingArguments(
    output_dir=str(BASE_DIR / 'models/distilbert_intent'),
    eval_strategy='epoch', save_strategy='epoch',
    learning_rate=2e-5, per_device_train_batch_size=16,
    per_device_eval_batch_size=16, num_train_epochs=4,
    weight_decay=0.01, logging_steps=20,
    load_best_model_at_end=True, metric_for_best_model='f1',
    report_to='none'
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=test_ds,
    processing_class=tokenizer, compute_metrics=compute_metrics
)

trainer.train()
trainer.save_model(str(BASE_DIR / 'models/distilbert_intent'))
tokenizer.save_pretrained(str(BASE_DIR / 'models/distilbert_intent'))
print('Intent model saved')

In [ ]:
# ── CELL 8: Evaluate Intent Classifier ───────────────────────────────
out   = trainer.predict(test_ds)
preds = np.argmax(out.predictions, axis=1)
true  = out.label_ids

acc       = accuracy_score(true, preds)
p, r, f,_ = precision_recall_fscore_support(true, preds, average='weighted', zero_division=0)

print(f'Accuracy : {acc:.4f}')
print(f'Precision: {p:.4f}')
print(f'Recall   : {r:.4f}')
print(f'F1       : {f:.4f}')
print()
print(classification_report(true, preds, target_names=le.classes_, zero_division=0))

# Save metrics
metrics = {'accuracy': float(acc), 'precision': float(p), 'recall': float(r), 'f1': float(f)}
with open(BASE_DIR / 'outputs/metrics/intent_metrics.json', 'w') as f_out:
    json.dump(metrics, f_out, indent=2)

# Confusion matrix
cm = confusion_matrix(true, preds)
plt.figure(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Intent Classifier Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(BASE_DIR / 'outputs/plots/intent_confusion_matrix.png', dpi=150)
plt.show()
print('Metrics saved')

In [ ]:
# ── CELL 9: Hard Real-World Test ─────────────────────────────────────
def predict_intent(query):
    inputs = tokenizer(query, return_tensors='pt', truncation=True, padding=True, max_length=64)
    model.eval()
    import torch
    with torch.no_grad():
        logits = model(**inputs).logits
    probs   = torch.softmax(logits, dim=1)[0]
    pred_id = torch.argmax(probs).item()
    return {'intent': id2label[pred_id], 'confidence': float(probs[pred_id])}

hard_tests = [
    ('I need somewhere quiet before my exam',          'recommend_place'),
    ('My laptop is not connecting to wifi',            'service_query'),
    ('Can I grab coffee before class?',                'menu_query'),
    ('Where do professors sit?',                       'faculty_query'),
    ('I misplaced my wallet',                          'lost_found'),
    ('I feel dizzy and need help',                     'emergency'),
    ('Is the library still open?',                     'ask_hours'),
    ('Anything happening on campus this week?',        'ask_event'),
    ('Tell me about the AI department',                'ask_department'),
    ('Where can I get bonafide certificate?',          'service_query'),
    ('Take me to the computer lab',                    'find_location'),
    ('What food can I get right now?',                 'menu_query'),
    ('How do I get from the library to the cafeteria?','find_location'),
    ('Price of tea',                                   'menu_query'),
    ('Who won the cricket match yesterday?',           'fallback'),
]

results = []
for query, true_intent in hard_tests:
    r = predict_intent(query)
    correct = r['intent'] == true_intent
    results.append({'query': query, 'true': true_intent, 'pred': r['intent'],
                    'conf': round(r['confidence'],3), 'correct': correct})

hard_df = pd.DataFrame(results)
print(hard_df.to_string(index=False))
print(f"\nHard test accuracy: {hard_df['correct'].mean():.2f}")

In [ ]:
# ── CELL 10: Build Semantic Retrieval Index ───────────────────────────
# Uses SentenceTransformer to encode KB text descriptions.
# At inference time, the user query is encoded and cosine similarity
# is used to find the best matching KB record.
from sentence_transformers import SentenceTransformer
import numpy as np, pickle

retrieval_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Build one rich text per KB record for encoding
def build_doc(loc):
    parts = [
        loc.get('name',''),
        loc.get('category','').replace('_',' '),
        loc.get('description',''),
        ' '.join(loc.get('keywords',[])),
        ' '.join(loc.get('aliases',[])),
    ]
    return ' '.join(p for p in parts if p)

docs       = [build_doc(loc) for loc in kb]
embeddings = retrieval_model.encode(docs, convert_to_numpy=True, show_progress_bar=True)

with open(BASE_DIR / 'models/retrieval_embeddings.pkl', 'wb') as f:
    pickle.dump({'embeddings': embeddings, 'kb_ids': [loc['id'] for loc in kb]}, f)

print(f'Embeddings shape: {embeddings.shape}')
print('Retrieval index saved')

In [ ]:
# ── CELL 11: Test Semantic Retrieval ─────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity

def semantic_retrieve(query, top_k=3):
    q_emb  = retrieval_model.encode([query])
    scores = cosine_similarity(q_emb, embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [{'record': kb[i], 'score': float(scores[i])} for i in top_idx]

test_queries = [
    'where is the library',
    'where can i print my assignment',
    'i lost my wallet',
    'where is civil department staff room',
    'price of tea',
    'i feel sick need help',
    'where can i study quietly',
]

print('=== SEMANTIC RETRIEVAL TEST ===')
for q in test_queries:
    results = semantic_retrieve(q, top_k=1)
    r = results[0]
    print(f'  Q: {q}')
    print(f'  A: {r["record"]["name"]} (score={r["score"]:.3f})')
    print()

In [ ]:
# ── CELL 12: Evaluate Top-1 and Top-3 Retrieval ───────────────────────
eval_set = [
    ('where is the library',                'central_library'),
    ('where can i print my assignment',     'printing_room'),
    ('i lost my wallet',                    'lost_and_found'),
    ('where is civil faculty room',         'civil_faculty_room'),
    ('i feel sick need help',               'medical_room'),
    ('where can i study quietly',           'reading_room'),
    ('where is cafeteria',                  'main_cafeteria'),
    ('where is computer lab',               'computer_lab'),
    ('where can i charge my laptop',        'charging_station'),
    ('where is the gym',                    'gym'),
]

top1 = top3 = 0
rows_eval = []
for q, expected_id in eval_set:
    results = semantic_retrieve(q, top_k=3)
    ids     = [r['record']['id'] for r in results]
    t1 = ids[0] == expected_id
    t3 = expected_id in ids
    top1 += int(t1)
    top3 += int(t3)
    rows_eval.append({'query': q, 'expected': expected_id,
                      'top1': ids[0], 'top1_correct': t1, 'top3_correct': t3})

eval_df = pd.DataFrame(rows_eval)
print(eval_df.to_string(index=False))
print(f'\nTop-1 Retrieval Accuracy: {top1/len(eval_set):.2f}')
print(f'Top-3 Retrieval Accuracy: {top3/len(eval_set):.2f}')

retrieval_metrics = {
    'top1_accuracy': top1/len(eval_set),
    'top3_accuracy': top3/len(eval_set)
}
with open(BASE_DIR / 'outputs/metrics/retrieval_metrics.json','w') as f:
    json.dump(retrieval_metrics, f, indent=2)
print('Retrieval metrics saved')

In [ ]:
# ── CELL 13: Summary Plot ─────────────────────────────────────────────
with open(BASE_DIR / 'outputs/metrics/intent_metrics.json') as f:
    im = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Intent metrics bar chart
intent_vals = [im['accuracy'], im['precision'], im['recall'], im['f1']]
axes[0].bar(['Accuracy','Precision','Recall','F1'], intent_vals, color=['#4C72B0','#DD8452','#55A868','#C44E52'])
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Intent Classifier Metrics')
for i, v in enumerate(intent_vals):
    axes[0].text(i, v+0.01, f'{v:.3f}', ha='center', fontsize=10)

# Retrieval accuracy bar chart
ret_vals = [retrieval_metrics['top1_accuracy'], retrieval_metrics['top3_accuracy']]
axes[1].bar(['Top-1','Top-3'], ret_vals, color=['#4C72B0','#55A868'])
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Semantic Retrieval Accuracy')
for i, v in enumerate(ret_vals):
    axes[1].text(i, v+0.01, f'{v:.2f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(BASE_DIR / 'outputs/plots/pipeline_metrics.png', dpi=150)
plt.show()
print('All training and evaluation complete.')
print('Models saved in models/ directory - you only need to run this notebook ONCE.')